# 06 - Does judge v2 agree with a human?

Run before the final run, not after. The ablation (B_rules, B_refusal, B_citations) is
graded by the judge alone, so if v2 is still unreliable those results are unusable,
we will need to know while the plan can still change.

v1 was lenient and unstable: on 2026-11-15 it marked all seven in-scope dev answers
`correct` (kappa 0.00) and changed its labels between two runs of the same answers.
v2 tightens the rubric, adds worked examples of `partial`, and takes the median of
three votes.

The check costs nothing new: the dev already runs and the manual grades already exist.

In [0]:
%pip install -q databricks-sdk
dbutils.library.restartPython()

In [0]:
# restartPython() resets sys.path
import sys
from pathlib import Path

repo_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

print("Repo root:", repo_root)

In [0]:
import csv

from src.evaluate import (EvalConfig, JUDGE_VERSION, agreement, evaluate_run,
                          format_summary, load_run, save_evaluations, summarize)

cfg = EvalConfig()
RUN_ID = "20261031T191325Z-09e6e4" # Variant A, dev split, run graded by hand

print("Judge:  ", cfg.judge_endpoint, "| version", JUDGE_VERSION, "|", cfg.n_votes, "votes")
print("Run:    ", RUN_ID)

## Load the run and the manual grades

In [0]:
rows = load_run(spark, cfg, RUN_ID)

user = spark.sql("SELECT current_user()").collect()[0][0]
manual_path = f"/Workspace/Users/{user}/manual_grading_{RUN_ID}.csv"
with open(manual_path, encoding="utf-8") as fh:
    manual = list(csv.DictReader(fh))

graded = sum(1 for m in manual if m["correctness"] or m["refusal"])
print(f"{len(rows)} answers, {len(manual)} manual rows, {graded} of them graded")

if graded < len(manual):
    print("WARNING: not every row is graded, the comparison below will be partial")

## Grade with v2

In [0]:
def progress(position, total, e):
    verdict = (e["judge_error"] or
               (f"{e['correctness']} / {e['groundedness']}" if e["in_scope"]
                else f"{e['refusal']}, invented_rule={e['invented_rule']}"))
    print(f"[{position}/{total}] {e['question_id']:<10} {verdict}")


evaluations = evaluate_run(spark, cfg, RUN_ID, on_progress=progress)

n_failed = sum(1 for e in evaluations if e["judge_error"])
print(f"\n{len(evaluations)} graded, {n_failed} judge errors")
print()
print(format_summary(summarize(evaluations)))

In [0]:
print("Saved to", save_evaluations(spark, cfg, evaluations))

## v2 against the manual grades
This is the number the decision rests on. kappa near 0 means the judge adds nothing beyond chance, however good the exact match may look.

In [0]:
for field, result in agreement(evaluations, manual).items():
    if not result["n"]:
        continue
    kappa = "-" if result["kappa"] is None else f"{result['kappa']:.2f}"
    print(f"{field:<14} n={result['n']:<3} exact={result['exact_match']:.2f}  kappa={kappa}")
    for d in result["disagreements"]:
        print(f"    {d['question_id']:<10} judge={d['judge']:<12} manual={d['manual']}")

## v1, v2 and the human, side by side
Rows saved before `judge_version` existed are v1. If several gradings of the same
version exist, the latest is used.

In [0]:
judged = [r.asDict() for r in spark.table(cfg.results_table)
          .filter(f"run_id = '{RUN_ID}'").collect()]

def latest(version):
    rows = [j for j in judged if (j.get("judge_version") or "v1") == version]
    out = {}
    for j in sorted(rows, key=lambda j: j["created_at"]):
        out[j["question_id"]] = j
    return out

v1, v2 = latest("v1"), latest("v2")
by_hand = {m["question_id"]: m for m in manual}

print(f"{'question':<10} {'field':<13} {'v1':<12} {'v2':<12} {'manual':<12} {'v2 ok?'}")
print("-" * 72)
for row in sorted(rows, key=lambda r: r["question_id"]):
    qid = row["question_id"]
    fields = ("correctness", "groundedness") if row["in_scope"] else ("refusal",)
    for field in fields:
        a = (v1.get(qid) or {}).get(field) or "-"
        b = (v2.get(qid) or {}).get(field) or "-"
        h = (by_hand.get(qid, {}).get(field) or "").strip().lower() or "-"
        mark = "" if h == "-" else ("yes" if b == h else "NO")
        print(f"{qid:<10} {field:<13} {a:<12} {b:<12} {h:<12} {mark}")

## What this means
Three things decide whether the ablation can rest on the judge:

1. **Spread.** If v2 gives every in-scope answer the same label it has no
   discriminative power, whatever the exact match says.
2. **kappa on correctness.** Below ~0.6 the judge's numbers need a caveat. Near 0 they
   cannot carry the ablation on their own.
3. **Direction of the disagreements.** All in one direction is a bias that can be
   described. Scattered is noise, which is worse.

If v2 fails any of these, say so in `analysis_plan.md` now: either the ablation is
reported with the judge's caveat attached, or those three variants get a small hand
check too. Both are allowed before the final run, not after.

In [0]:
in_scope = [e for e in evaluations if e["in_scope"] and not e["judge_error"]]
labels = [e["correctness"] for e in in_scope]

print("correctness labels given by v2:", {l: labels.count(l) for l in set(labels)})
print("spread:", "none, judge says the same thing every time" if len(set(labels)) == 1
      else f"{len(set(labels))} different labels")

# How often the three votes agreed with each other. 
# Low values are the v1 instability still showing, 
# independently of whether the human agrees.
shares = [e["judge_agreement"] for e in evaluations if e.get("judge_agreement") is not None]
if shares:
    print()
    print(f"vote agreement: mean {sum(shares)/len(shares):.2f}, min {min(shares):.2f}")
    for e in sorted(evaluations, key=lambda e: e.get("judge_agreement") or 1):
        if e.get("judge_agreement") is not None and e["judge_agreement"] < 1.0:
            print(f"   {e['question_id']:<10} votes agreed {e['judge_agreement']:.2f}")